# 🧪 W3-D7 概念实验：大模型训练全景（预训练 → SFT → RLHF/DPO）

> 配套阅读：`第3周-Day7-第三周总复习.md`（14 张知识卡片与测试题在那边）
> 复习日做 5 个实验，把第三周的三级火箭**真的点一次火**：
> **不同阶段的 Loss 长什么样 / 奖励模型怎么从偏好数据学出来 / DPO 的隐式奖励 /
> 学习率怎么退火 / Scaling Law 与涌现**
>
> 实验环境：纯 numpy + matplotlib。

## 实验 1：预训练 Loss vs SFT Loss —— 只差一个 mask

同一组 logits：**预训练**对每个位置都算"下一词"交叉熵（读什么就学什么）；
**SFT** 只对回答段算损失——模型不需要学会复述用户的问题。

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
T, V = 6, 5                                  # 6 个位置，词表 5
logits = rng.normal(size=(T, V))             # 模型未归一化输出
labels = rng.integers(0, V, size=T)          # 每个位置的正确下一 token
prompt_mask = np.array([0, 0, 0, 1, 1, 1])   # 前 3 个位置是指令，后 3 个是回答

def masked_ce(logits, labels, mask):
    """带掩码的交叉熵：只统计 mask=1 的位置"""
    z = logits - logits.max(axis=1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    sel = mask.astype(bool)
    return -logp[sel, labels[sel]].mean()

pretrain_loss = masked_ce(logits, labels, np.ones(T, dtype=int))
sft_loss = masked_ce(logits, labels, prompt_mask)

print(f"预训练 Loss（全部 6 个位置）: {pretrain_loss:.4f}")
print(f"  （均匀分布的理论底线 ln(5)={np.log(5):.2f}；随机 logits 更高，训练就是把它压下去）")
print(f"SFT Loss（只算回答段 3 个位置）: {sft_loss:.4f}")
print("\n→ 同一个模型、同一份数据，训练目标由 mask 决定：")
print("   预训练 = 学语言本身；SFT = 学对话格式与回答方式（不复述问题）")
print("   RLHF/DPO = 再对齐人类偏好（接下来两个实验）")

## 实验 2：RLHF 第一步 —— 用 Bradley-Terry 从偏好对学出奖励模型

人类说"A 比 B 好"，BT 模型把它变成概率：`P(A≻B) = σ(r_A − r_B)`。
用 numpy 手写 SGD，从 256 对模拟偏好里拟合 8 个回答的奖励值，
看它能否恢复真实的偏好排序（rank 相关 & 留出集准确率）。

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))

# 真实奖励（模拟人类真实偏好强度），训练时不可见
items = 8
r_true = np.linspace(-1.5, 1.5, items)

# 采样偏好对：胜者按 BT 概率随机
n_pairs = 256
a_idx = rng.integers(0, items, n_pairs)
b_idx = rng.integers(0, items, n_pairs)
keep = a_idx != b_idx
a_idx, b_idx = a_idx[keep], b_idx[keep]
win_is_a = rng.random(len(a_idx)) < sigmoid(r_true[a_idx] - r_true[b_idx])

# SGD 拟合奖励模型
r_hat = np.zeros(items)
lr = 0.05
for epoch in range(3000):
    m = r_hat[a_idx] - r_hat[b_idx]
    s = sigmoid(m)
    dL_dm = np.where(win_is_a, -(1 - s), s)      # -log σ(±m) 的梯度
    grad = np.zeros(items)
    np.add.at(grad, a_idx, dL_dm)
    np.add.at(grad, b_idx, -dL_dm)
    r_hat -= lr * grad / len(a_idx)

def ranks(x):
    order = np.argsort(x); r = np.empty(len(x)); r[order] = np.arange(len(x)); return r
rank_corr = np.corrcoef(ranks(r_hat), ranks(r_true))[0, 1]

# 留出集：200 对新偏好，用学到的 r_hat 预测
a2 = rng.integers(0, items, 200); b2 = rng.integers(0, items, 200)
keep2 = a2 != b2; a2, b2 = a2[keep2], b2[keep2]
pred_win = r_hat[a2] > r_hat[b2]
true_win = rng.random(len(a2)) < sigmoid(r_true[a2] - r_true[b2])
acc = (pred_win == true_win).mean()

print("真实奖励:  ", r_true.round(2))
print("拟合奖励:  ", r_hat.round(2), "（只差一个常数偏移，排序一致即可）")
print(f"\nSpearman 秩相关: {rank_corr:.3f}（1.0 = 完美恢复排序）")
print(f"留出集偏好预测准确率: {acc:.1%}（随机基线 50%）")
print("\n→ 这就是 RLHF 的奖励模型：偏好对 → 可打分的 RM → 再用 PPO 优化策略")

## 实验 3：DPO —— 不训 RM、不做强化学习的"隐式奖励"

DPO 直接对偏好对优化策略：`L = −log σ(β·[(logπ_w−logπ_ref_w) − (logπ_l−logπ_ref_l)])`。
模拟策略逐步偏离参考（θ 从 0 到 3）：偏好 margin 增大、loss 下降、KL 也在涨——
**β 就是踩油门的力度：太小收敛慢，太大容易偏离参考模型失去通用性**。

In [ ]:
beta_lo, beta_hi = 0.1, 0.3
logp_ref_w, logp_ref_l = -1.0, -1.5      # 参考模型对优选/劣选回答的 log 概率

print(f"{'偏离量θ':>8}{'margin':>9}{'DPO loss β=0.1':>16}{'DPO loss β=0.3':>16}{'KL(近似)':>10}")
for theta in [0.0, 0.5, 1.0, 2.0, 3.0]:
    # 策略：优选回答概率上升 1.2θ，劣选下降 0.8θ（模拟被 DPO 优化）
    margin = (1.2 * theta) - (-0.8 * theta)
    loss_lo = -np.log(sigmoid(beta_lo * margin))
    loss_hi = -np.log(sigmoid(beta_hi * margin))
    kl = 0.5 * theta**2                    # 与参考策略的 KL 随偏离平方增长
    print(f"{theta:>8.1f}{margin:>9.1f}{loss_lo:>16.4f}{loss_hi:>16.4f}{kl:>10.2f}")

print("\n→ θ 增大：margin 增大、loss 下降，但 KL 同步上升（过拟合偏好、丢通用能力）")
print("→ DPO 没有 PPO 那个显式 KL 惩罚项，全靠 β 控制节奏 —— 简单但更要小心调")

## 实验 4：学习率调度 —— warmup + cosine 是标配

训练初期梯度乱（warmup 保护），中后期余弦退火让参数落进更平的极小值。
画三种调度对比：warmup+cosine / 恒定 / 阶梯衰减。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

steps, warmup, peak = 1000, 100, 3e-4
t = np.arange(1, steps + 1)

warm_cos = peak * np.minimum(t / warmup, 1.0) *            0.5 * (1 + np.cos(np.pi * np.clip((t - warmup) / (steps - warmup), 0, 1)))
constant = peak * np.ones(steps)
step_decay = peak * 0.1 ** (t // 333)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(t, warm_cos, label="warmup + cosine（主流标配）")
ax.plot(t, constant, label="恒定学习率")
ax.plot(t, step_decay, label="阶梯衰减")
ax.set_xlabel("训练步数"); ax.set_ylabel("学习率")
ax.set_title("三种学习率调度（峰值 3e-4，warmup 100 步）")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"warmup+cosine 覆盖面积（平均学习率）: {warm_cos.mean()/peak:.1%} × 峰值")
print("→ 先爬坡再退火：前期稳、后期精，比恒定/阶梯更适合大模型的长训练")

## 实验 5：Scaling Law 与"涌现" —— 平滑下降 + 阈值度量 = 突现

Chinchilla 式幂律：`L(N) = E + A·N^(−α)`，loss 随参数**平滑**下降；
但如果能力指标是"超过阈值才算对"（如多位数乘法），
平滑的 loss 会在某个规模附近表现为能力的**突跳**——涌现是度量方式的产物。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

N = np.logspace(7, 11.24, 300)          # 1e7 ~ 1.7e11 参数
loss = 1.7 + 20 * N**-0.1               # 幂律 + 不可约底
capability = 1 / (1 + np.exp(-(3.4 - loss) / 0.06))   # 阈值型能力指标

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].loglog(N, loss)
axes[0].set_xlabel("参数量 N"); axes[0].set_ylabel("验证 Loss")
axes[0].set_title("Scaling Law：loss 平滑下降（无跳变）")
axes[0].axhline(3.4, ls="--", color="gray"); axes[0].text(2e7, 3.45, "能力阈值 L0=3.4")
axes[1].semilogx(N, capability * 100)
axes[1].set_xlabel("参数量 N"); axes[1].set_ylabel("能力通过率 (%)")
axes[1].set_title("同一模型、阈值型指标：能力「突现」")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

cross = N[np.argmax(capability > 0.5)]
print(f"能力过半的参数量 ≈ {cross/1e9:.0f}B —— loss 从未跳变，跳的是度量方式")
print("\n第三周全景: 预训练(吃语料) → SFT(学格式) → RM(吃偏好) → PPO/DPO(对齐)，")
print("再加两条工程线: 学习率调度(实验4) 与 规模律(实验5)")

## 结论

| 阶段 | 实验 | 关键结论 |
|---|---|---|
| 预训练 vs SFT | 1 | 同一模型，mask 决定训练目标 |
| RLHF 奖励模型 | 2 | BT 模型 + 256 对偏好 → 秩相关 0.95、留出集 74%（随机 50%） |
| DPO | 3 | 隐式奖励直改策略，β 控制偏离速度 |
| 调度 | 4 | warmup+cosine 是长训练标配 |
| 规模律 | 5 | loss 平滑、能力突现：涌现=阈值度量 |

→ 深入阅读：同目录 `.md` 版本第二节（完整数据流 + 各阶段 Loss 对比表）